# Trust List 2 v013 — TRUST5

This kernel applies the frozen five-checkpoint TRUST5 score rule. It loads the checkpoints sequentially, writes `submission.parquet` with exactly `id,model`, and does not retrain or reselect models.


In [ ]:
from pathlib import Path
import gc, json, os, subprocess, sys, time
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
INPUT_ROOT = Path("/kaggle/input")
matches = list(INPUT_ROOT.rglob("trust2_v013_top5_manifest.json"))
assert len(matches) == 1, f"expected one frozen TRUST5 manifest, got {matches}"
DATA = matches[0].parent
BUNDLE = json.loads(matches[0].read_text())
assert BUNDLE["trust_list_version"] == "trust-list-2-v013"
assert BUNDLE["sequence_ids"] == ["N0071", "N0056", "N0005", "N0058", "N0072"]
assert len(BUNDLE["candidates"]) == 5 and len(BUNDLE["rules"]) == 89
from importlib.metadata import version
SKLEARN_VERSION = "1.9.1"
if version("scikit-learn") != SKLEARN_VERSION:
    wheel = next(INPUT_ROOT.rglob("scikit_learn-1.9.1-*.whl"), None)
    assert wheel is not None, "pinned sklearn wheel missing"
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--force-reinstall", str(wheel)], check=True)
import joblib, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "Kaggle GPU is unavailable"
WORK = Path("/kaggle/working/trust2_v013_top5_runtime")
SRC = WORK / "src"
SRC.mkdir(parents=True, exist_ok=True)
for name in ("__init__.py", "data_factory.py", "feature_factory.py", "model_factory.py", "inference.py"):
    (SRC / name).symlink_to(DATA / f"source__{name}")
sys.path.insert(0, str(WORK))
CHECKPOINTS = WORK / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
for sequence_id, candidate in BUNDLE["candidates"].items():
    dest = CHECKPOINTS / sequence_id
    dest.mkdir(parents=True, exist_ok=True)
    for item in candidate["files"]:
        source = DATA / item["flat_filename"]
        assert source.is_file(), f"checkpoint asset missing: {sequence_id}/{item['flat_filename']}"
        target = dest / item["relative_path"]
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
    candidate["runtime_checkpoint"] = str(dest if candidate["checkpoint_type"] == "directory" else dest / candidate["files"][0]["relative_path"])

granite_manifest = next(INPUT_ROOT.rglob("zod03_manifest.json"), None)
assert granite_manifest is not None, "the established ZOD03 base assets are not attached"
ZOD03_DATA = granite_manifest.parent
ZOD03 = json.loads(granite_manifest.read_text())
granite_candidate = next(c for c in BUNDLE["candidates"].values() if c["model"] == "granite4_h_micro")
granite_base = next((c for c in ZOD03["candidates"].values()
                     if c.get("base_model") == granite_candidate["base_model"]
                     and c.get("base_model_revision") == granite_candidate["base_model_revision"]
                     and c.get("base_files")), None)
assert granite_base is not None, "exact Granite base revision not found in established ZOD03 assets"
BASES = {}
granite_dir = WORK / "bases" / "granite4_h_micro"
granite_dir.mkdir(parents=True, exist_ok=True)
for item in granite_base["base_files"]:
    source = ZOD03_DATA / item["flat_filename"]
    target = granite_dir / item["relative_path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    target.symlink_to(source)
BASES["granite4_h_micro"] = granite_dir

def resolve_model(marker):
    found = [path.parent for path in INPUT_ROOT.rglob("config.json") if marker.lower() in str(path.parent).lower()]
    assert found, f"attached Kaggle model not found: {marker}"
    depth = min(len(path.parts) for path in found)
    roots = sorted({path for path in found if len(path.parts) == depth})
    assert len(roots) == 1, f"expected one model root for {marker}, found {roots}"
    return roots[0]
BASES["ministral3_3b"] = resolve_model("ministral-3-3b-base-2512")
# Kaggle currently ships TorchAO 0.10.0, while this PEFT version requires >0.16.
# TorchAO is optional here; force PEFT to use its standard LoRA module dispatcher.
import peft.tuners.lora.torchao as peft_torchao
peft_torchao.is_torchao_available = lambda: False
from src.inference import _predict_neural, predict_catboost_stylometry, predict_svc_bundle
TEST = next((path for path in (INPUT_ROOT / "competitions/name-that-llm/test.csv", INPUT_ROOT / "name-that-llm/test.csv") if path.is_file()), None)
assert TEST is not None, "competition test.csv not found"


In [ ]:
test = pd.read_csv(TEST, dtype={"id": "string"})
assert {"id", "text"}.issubset(test.columns)
assert test.id.notna().all() and not test.id.duplicated().any()
test.text = test.text.fillna("").astype(str)
classes = list(map(str, BUNDLE["classes"]))
texts = test.text.tolist()
def predict_checkpoint(sequence_id):
    candidate = BUNDLE["candidates"][sequence_id]
    model, checkpoint = candidate["model"], Path(candidate["runtime_checkpoint"])
    if model == "svc":
        probabilities, labels = predict_svc_bundle(joblib.load(checkpoint), texts)
    elif model == "catboost_stylometry":
        probabilities, labels = predict_catboost_stylometry(checkpoint, texts, classes,
            temperature=float(candidate.get("calibration_temperature", 1.0)))
    else:
        base = BASES.get(model)
        assert model == "ultrabert380" or base is not None, f"model base not resolved for {model}"
        batch_size = min(4, max(1, int(candidate["config"].get("inference_batch_size", 4) or 4)))
        while True:
            try:
                probabilities, labels = _predict_neural(
                    model, checkpoint, texts, classes, batch_size=batch_size,
                    max_length=int(candidate["config"].get("max_length", 384)),
                    temperature=float(candidate.get("calibration_temperature", 1.0)),
                    base_checkpoint=str(base) if base else candidate.get("base_model"),
                    revision=None if base else candidate.get("model_revision"),
                    fix_mistral_regex=bool(candidate["config"].get("fix_mistral_regex", False)),
                    text_only_backbone=bool(candidate["config"].get("text_only_backbone", False)),
                    quantization=candidate["config"].get("quantization"),
                    attention_backend=candidate["config"].get("attention_backend"))
                break
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower() or batch_size <= 1:
                    raise
                batch_size = max(1, batch_size // 2)
                gc.collect(); torch.cuda.empty_cache()
    labels = list(map(str, labels))
    order = [labels.index(label) for label in classes]
    probabilities = np.asarray(probabilities, dtype=np.float64)[:, order]
    assert probabilities.shape == (len(test), 18) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)
    top1 = np.asarray(classes, dtype=object)[probabilities.argmax(axis=1)]
    cache_dir = Path("/kaggle/working/checkpoint_probabilities")
    cache_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache_dir / f"{sequence_id}.npz", id=test.id.astype(str).to_numpy(), probabilities=probabilities, classes=np.asarray(classes))
    del probabilities
    gc.collect(); torch.cuda.empty_cache()
    return top1

started = time.time()
top1 = {sequence_id: predict_checkpoint(sequence_id) for sequence_id in BUNDLE["sequence_ids"]}
family_by_sequence = {key: value["model"].split("_")[0] for key, value in BUNDLE["candidates"].items()}
# Use the same model-family aliases and family divisors as the local subset implementation.
family_by_sequence.update({"N0005": "catboost", "N0056": "granite", "N0058": "mistral", "N0071": "ultrabert", "N0072": "svc"})
family_counts = {family: sum(value == family for value in family_by_sequence.values()) for family in set(family_by_sequence.values())}
rule_by_key = {(rule["sequence_id"], rule["label"]): rule for rule in BUNDLE["rules"]}
scores = np.zeros((len(test), len(classes)), dtype=np.float64)
label_index = {label: index for index, label in enumerate(classes)}
for sequence_id in BUNDLE["sequence_ids"]:
    family_divisor = family_counts[family_by_sequence[sequence_id]]
    for row_index, label in enumerate(top1[sequence_id]):
        rule = rule_by_key.get((sequence_id, str(label)))
        if rule is None:
            continue
        support = int(rule["prediction_count"])
        contribution = float(rule["precision"]) * support / (support + 30)
        scores[row_index, label_index[str(label)]] += contribution / family_divisor
prediction = np.asarray(classes, dtype=object)[scores.argmax(axis=1)]
no_votes = scores.sum(axis=1) == 0
fallback = BUNDLE["fallback_sequence_id"]
prediction[no_votes] = top1[fallback][no_votes]
submission = pd.DataFrame({"id": test.id.astype(str), "model": prediction.astype(str)})
assert submission.columns.tolist() == ["id", "model"]
assert submission.id.tolist() == test.id.astype(str).tolist()
assert submission.id.nunique() == len(test) and submission.model.notna().all()
assert submission.model.isin(classes).all()
submission.to_parquet("/kaggle/working/submission.parquet", index=False)
print({"algorithm": BUNDLE["algorithm"], "trust_list_version": BUNDLE["trust_list_version"],
       "checkpoint_ids": BUNDLE["sequence_ids"], "rows": len(submission),
       "fallback_rows": int(no_votes.sum()), "runtime_seconds": round(time.time()-started, 1),
       "schema": submission.columns.tolist()})
display(submission.head())
# Reuse the exact same five test-probability matrices for EQUAL5 and WEIGHTED5.
probability_cache = {}
for sequence_id in BUNDLE["sequence_ids"]:
    cached = np.load(Path("/kaggle/working/checkpoint_probabilities") / f"{sequence_id}.npz", allow_pickle=True)
    assert cached["id"].astype(str).tolist() == test.id.astype(str).tolist()
    assert cached["classes"].astype(str).tolist() == classes
    probability_cache[sequence_id] = cached["probabilities"].astype(np.float64)

EQUAL5_WEIGHTS = {sequence_id: 0.2 for sequence_id in BUNDLE["sequence_ids"]}
WEIGHTED5_WEIGHTS = {
    "N0071": 0.2811918866145629,
    "N0056": 0.23247150955573173,
    "N0005": 0.18335267365939653,
    "N0058": 0.23454557290930692,
    "N0072": 0.0684383572610019,
}
assert list(EQUAL5_WEIGHTS) == BUNDLE["sequence_ids"]
assert list(WEIGHTED5_WEIGHTS) == BUNDLE["sequence_ids"]
assert abs(sum(WEIGHTED5_WEIGHTS.values()) - 1.0) < 1e-12

def export_probability_blend(filename, weights):
    combined = sum(probability_cache[key] * float(weights[key]) for key in BUNDLE["sequence_ids"])
    prediction = np.asarray(classes, dtype=object)[combined.argmax(axis=1)]
    output = pd.DataFrame({"id": test.id.astype(str), "model": prediction.astype(str)})
    assert output.columns.tolist() == ["id", "model"]
    assert output.id.tolist() == test.id.astype(str).tolist()
    assert output.id.nunique() == len(test) and output.model.notna().all()
    assert output.model.isin(classes).all()
    output.to_parquet(f"/kaggle/working/{filename}", index=False)
    print({"strategy": filename.removeprefix("submission_").removesuffix(".parquet"),
           "rows": len(output), "checkpoint_ids": BUNDLE["sequence_ids"], "weights": weights})

export_probability_blend("submission_equal5.parquet", EQUAL5_WEIGHTS)
export_probability_blend("submission_weighted5.parquet", WEIGHTED5_WEIGHTS)
# This competition requires its primary output to be named exactly submission.parquet.
ACTIVE_SUBMISSION_STRATEGY = "WEIGHTED5"
assert ACTIVE_SUBMISSION_STRATEGY in {"EQUAL5", "WEIGHTED5"}
from shutil import copyfile
copyfile(f"/kaggle/working/submission_{ACTIVE_SUBMISSION_STRATEGY.lower()}.parquet",
         "/kaggle/working/submission.parquet")
active_submission = pd.read_parquet("/kaggle/working/submission.parquet")
assert active_submission.columns.tolist() == ["id", "model"]
assert active_submission.id.tolist() == test.id.astype(str).tolist()
print({"active_submission_strategy": ACTIVE_SUBMISSION_STRATEGY,
       "required_output": "/kaggle/working/submission.parquet"})


